# Full RAG + Uncertainty Estimation Pipeline (FIXED)
### IR Project — Arnes HPC Jupyter Environment

**Models Supported:**
- **Generator LLM:** Qwen2.5-7B-Instruct (default) or Falcon-7B-Instruct
- **Retriever:** BM25 + Cross-Encoder Reranker (ms-marco-MiniLM-L6-v2)
- **Evaluation:** SAFE (using local Qwen/Falcon as rater)

**Uncertainty Estimation Methods:**
- **MARS (White-box):** Model confidence via token-level entropy/NLL
- **Eccentricity (Black-box):** Embedding-based answer-context distance
- **BONUS - Semantic Entropy (Hybrid):** Cluster-based semantic uncertainty

---
## How to Switch Between Qwen and Falcon:
```python
USE_FALCON = True   # Use Falcon-7B-Instruct
USE_FALCON = False  # Use Qwen2.5-7B-Instruct (default)
```

**Note:** This version includes smart model loading that:
1. First tries to load from local HF cache
2. If that fails, downloads fresh and overwrites

## 1. Environment Configuration

In [2]:
# ============================================================================
# CELL 1: HuggingFace Cache Configuration (MUST BE FIRST CELL)
# ============================================================================
import os

# HuggingFace cache redirection - adjust paths for your cluster
os.environ["HF_HOME"] = "/d/hpc/projects/FRI/ma76193/hf_home"
os.environ["HF_DATASETS_CACHE"] = "/d/hpc/projects/FRI/ma76193/hf_datasets"
os.environ["TRANSFORMERS_CACHE"] = "/d/hpc/projects/FRI/ma76193/hf_transformers"

for path in [os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"], os.environ["TRANSFORMERS_CACHE"]]:
    os.makedirs(path, exist_ok=True)

print(f"HF_HOME = {os.environ['HF_HOME']}")
print(f"HF_DATASETS_CACHE = {os.environ['HF_DATASETS_CACHE']}")
print(f"TRANSFORMERS_CACHE = {os.environ['TRANSFORMERS_CACHE']}")

HF_HOME = /d/hpc/projects/FRI/ma76193/hf_home
HF_DATASETS_CACHE = /d/hpc/projects/FRI/ma76193/hf_datasets
TRANSFORMERS_CACHE = /d/hpc/projects/FRI/ma76193/hf_transformers


In [3]:
# ============================================================================
# CELL 2: Java Configuration & Logging Setup
# ============================================================================
import os, sys, json, time, random, logging, threading, subprocess
from pathlib import Path
from datetime import datetime

# Java configuration for Pyserini
JAVA = "/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12"
os.environ["JAVA_HOME"] = JAVA
os.environ["JVM_PATH"] = f"{JAVA}/lib/server/libjvm.so"
os.environ["LD_LIBRARY_PATH"] = f"{JAVA}/lib/server:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["PATH"] = f"{JAVA}/bin:" + os.environ["PATH"]

print(f"JAVA_HOME = {os.environ['JAVA_HOME']}")
!java -version

import torch

# Logging
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / "full_notebook.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8"), logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("nb")

def banner(msg):
    log.info("=" * 80)
    log.info(f"*** {msg} ***")
    log.info("=" * 80)

JAVA_HOME = /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12
openjdk version "21.0.1" 2023-10-17 LTS
OpenJDK Runtime Environment Temurin-21.0.1+12 (build 21.0.1+12-LTS)
OpenJDK 64-Bit Server VM Temurin-21.0.1+12 (build 21.0.1+12-LTS, mixed mode, sharing)


In [4]:
# ============================================================================
# CELL 3: Heartbeat & Environment Optimization
# ============================================================================
_stop_hb = threading.Event()

def _heartbeat(period=30):
    while not _stop_hb.is_set():
        log.info("[HEARTBEAT] Notebook alive...")
        time.sleep(period)

hb = threading.Thread(target=_heartbeat, daemon=True)
hb.start()

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("JAVA_TOOL_OPTIONS", "-Xms1g -Xmx8g")

log.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        log.info(f"GPU[{i}] {torch.cuda.get_device_name(i)}")

2025-12-09 20:43:52,067 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:43:52,192 | INFO | CUDA available: True
2025-12-09 20:43:52,223 | INFO | GPU[0] Tesla V100S-PCIE-32GB
2025-12-09 20:43:52,224 | INFO | GPU[1] Tesla V100S-PCIE-32GB


In [5]:
# ============================================================================
# CELL 4: Directory Setup
# ============================================================================
FULL_SHARDS_DIR = Path("data/wiki18/shards_full")
FULL_INDEX_DIR = Path("index/bm25_full")
RUNS = Path("runs")
RUNS.mkdir(exist_ok=True)

RETR_DIR = RUNS / "retrieval"
ANS_DIR = RUNS / "answers"
UE_DIR = RUNS / "ue"
REPORTS_DIR = Path("reports")

for p in [RETR_DIR, ANS_DIR, UE_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

banner("Directories prepared")

2025-12-09 20:43:52,485 | INFO | ================================================================================
2025-12-09 20:43:52,485 | INFO | *** Directories prepared ***
2025-12-09 20:43:52,486 | INFO | ================================================================================


## 2. Data Preparation: Wiki-18 Corpus

In [6]:
# ============================================================================
# CELL 5: Download Wiki-18 Dataset
# ============================================================================
from huggingface_hub import snapshot_download
import gzip

HF_DATA_REPO = "PeterJinGo/wiki-18-corpus"

def download_wiki18():
    banner("DOWNLOAD WIKI-18")
    path = snapshot_download(repo_id=HF_DATA_REPO, repo_type="dataset", allow_patterns=["*.jsonl.gz"], local_files_only=False)
    candidates = list(Path(path).rglob("*.jsonl.gz"))
    if not candidates:
        raise RuntimeError("No wiki18 .jsonl.gz found")
    return candidates[0]

wiki_gz = download_wiki18()
log.info(f"wiki gz: {wiki_gz}")

2025-12-09 20:43:53,471 | INFO | ================================================================================
2025-12-09 20:43:53,472 | INFO | *** DOWNLOAD WIKI-18 ***
2025-12-09 20:43:53,473 | INFO | ================================================================================
2025-12-09 20:43:54,098 | INFO | wiki gz: /d/hpc/projects/FRI/ma76193/hf_home/hub/datasets--PeterJinGo--wiki-18-corpus/snapshots/69c1c00ffe7c5554c68d8548355cb22e46aabc51/wiki-18.jsonl.gz


In [7]:
# ============================================================================
# CELL 6: Shard Wiki-18 & Build BM25 Index
# ============================================================================
def shard_wiki18(gz_path, out_dir, shard_size=100_000):
    banner("SHARDING WIKI-18")
    out_dir.mkdir(parents=True, exist_ok=True)
    if list(out_dir.glob("*.jsonl")):
        log.info(f"Shards exist. Skipping.")
        return
    total, idx, written = 0, 0, 0
    out_f = open(out_dir / f"wiki18_full_{idx:03d}.jsonl", "w", encoding="utf-8")
    with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except:
                continue
            text = obj.get("text") or obj.get("contents") or ""
            if not text.strip():
                continue
            rec = {"id": obj.get("id") or obj.get("page_id") or str(total), "contents": text.strip()}
            out_f.write(json.dumps(rec) + "\n")
            total += 1
            written += 1
            if written >= shard_size:
                out_f.close()
                idx += 1
                written = 0
                out_f = open(out_dir / f"wiki18_full_{idx:03d}.jsonl", "w", encoding="utf-8")
    out_f.close()
    log.info(f"Sharding complete. Total: {total}")

def build_index(input_dir, index_dir, threads=8):
    banner("BM25 INDEX BUILD")
    if index_dir.exists() and any(index_dir.iterdir()):
        log.info("Index exists. Skipping.")
        return
    cmd = [sys.executable, "-m", "pyserini.index.lucene", "--collection", "JsonCollection",
           "--input", str(input_dir), "--index", str(index_dir), "--generator", "DefaultLuceneDocumentGenerator",
           "--threads", str(threads), "--storePositions", "--storeDocvectors", "--storeRaw"]
    subprocess.run(cmd, check=True)

shard_wiki18(wiki_gz, FULL_SHARDS_DIR)
build_index(FULL_SHARDS_DIR, FULL_INDEX_DIR)

2025-12-09 20:43:54,111 | INFO | ================================================================================
2025-12-09 20:43:54,112 | INFO | *** SHARDING WIKI-18 ***
2025-12-09 20:43:54,113 | INFO | ================================================================================
2025-12-09 20:43:54,116 | INFO | Shards exist. Skipping.
2025-12-09 20:43:54,116 | INFO | ================================================================================
2025-12-09 20:43:54,117 | INFO | *** BM25 INDEX BUILD ***
2025-12-09 20:43:54,117 | INFO | ================================================================================
2025-12-09 20:43:54,119 | INFO | Index exists. Skipping.


## 3. Query Sampling

In [8]:
# ============================================================================
# CELL 7: Sample Queries
# ============================================================================
def extract_query_text(obj):
    for k in ["query", "question", "prompt", "instruction", "text", "claim"]:
        if k in obj and isinstance(obj[k], str) and obj[k].strip():
            return obj[k].strip()
    for v in obj.values():
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None

def sample_queries(src_path, out_path, n, seed):
    banner("SAMPLE QUERIES")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        log.info(f"Exists: {out_path}")
        return out_path
    items = []
    with open(src_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            try:
                obj = json.loads(line)
            except:
                continue
            q = extract_query_text(obj)
            if q:
                qid = str(obj.get("id") or obj.get("_id") or f"s{i:08d}")
                items.append((qid, q))
    random.Random(seed).shuffle(items)
    with open(out_path, "w", encoding="utf-8") as f:
        for i, (qid, q) in enumerate(items[:n], 1):
            f.write(json.dumps({"id": f"b{i:03d}", "orig_id": qid, "query": q}, ensure_ascii=False) + "\n")
    log.info(f"Wrote {n} queries -> {out_path}")
    return out_path

QUERY_SRC = Path("data/queries/factscore_bio.jsonl")
SAMPLED_QUERIES = Path("data/queries/notebook.seed1337.jsonl")
sample_queries(QUERY_SRC, SAMPLED_QUERIES, n=50, seed=1337)

2025-12-09 20:43:54,795 | INFO | ================================================================================
2025-12-09 20:43:54,795 | INFO | *** SAMPLE QUERIES ***
2025-12-09 20:43:54,796 | INFO | ================================================================================
2025-12-09 20:43:54,797 | INFO | Exists: data/queries/notebook.seed1337.jsonl


PosixPath('data/queries/notebook.seed1337.jsonl')

## 4. Retrieval: BM25 + Cross-Encoder Reranking

In [9]:
# ============================================================================
# CELL 8: BM25 + Cross-Encoder Reranking
# ============================================================================
from pyserini.search.lucene import LuceneSearcher
from sentence_transformers import CrossEncoder

def get_doc_text(raw):
    try:
        return json.loads(raw).get("contents", "").strip()
    except:
        return str(raw).strip()

def retrieve_rerank(queries_path, index_dir, out_path, k_first=1000, k_keep=3,
                    ce_model="cross-encoder/ms-marco-MiniLM-L6-v2", batch_size=64):
    banner("BM25 + CROSS-ENCODER RERANK")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    done = {}
    if out_path.exists():
        with open(out_path, "r") as f:
            for line in f:
                try:
                    ex = json.loads(line)
                    done[ex["id"]] = ex
                except:
                    pass
        log.info(f"Resume: {len(done)} done")
    try:
        ce = CrossEncoder(ce_model, device="cuda" if torch.cuda.is_available() else "cpu")
    except:
        ce = None
    searcher = LuceneSearcher(str(index_dir))
    searcher.set_bm25(k1=0.9, b=0.4)
    new = 0
    with open(queries_path, "r") as fin, open(out_path, "a") as fout:
        for line in fin:
            ex = json.loads(line)
            qid, query = ex["id"], ex["query"]
            if qid in done:
                continue
            hits = searcher.search(query, k_first)
            if not hits:
                fout.write(json.dumps({"id": qid, "query": query, "docs": []}) + "\n")
                new += 1
                continue
            candidates = [{"docid": h.docid, "raw": searcher.doc(h.docid).raw(),
                          "bm25": float(h.score), "text": get_doc_text(searcher.doc(h.docid).raw())} for h in hits]
            if ce:
                try:
                    scores = ce.predict([(query, c["text"]) for c in candidates], batch_size=batch_size)
                    for c, s in zip(candidates, scores):
                        c["ce"] = float(s)
                    candidates.sort(key=lambda x: x.get("ce", -1e9), reverse=True)
                except:
                    candidates.sort(key=lambda x: x["bm25"], reverse=True)
            else:
                candidates.sort(key=lambda x: x["bm25"], reverse=True)
            final_docs = [{"docid": c["docid"], "raw": c["raw"], "bm25": c["bm25"], "ce": c.get("ce")} for c in candidates[:k_keep]]
            fout.write(json.dumps({"id": qid, "query": query, "docs": final_docs}) + "\n")
            new += 1
            log.info(f"[{qid}] {len(final_docs)} docs")
    log.info(f"Rerank done: {new} new")
    return out_path

RETR_FILE = RETR_DIR / "notebook.seed1337.rerank3.jsonl"
retrieve_rerank(SAMPLED_QUERIES, FULL_INDEX_DIR, RETR_FILE, k_first=1000, k_keep=3)

2025-12-09 20:43:56,755 | INFO | 
Using override env var JVM_PATH (/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12/lib/server/libjvm.so) to load libjvm.
Please report your system information (os version, java
version, etc), and the path that works for you, to the
PyJNIus project, at https://github.com/kivy/pyjnius/issues.
so we can improve the automatic discovery.



Picked up JAVA_TOOL_OPTIONS: -Xms1g -Xmx8g


2025-12-09 20:44:04,177 | INFO | Loading faiss with AVX2 support.
2025-12-09 20:44:04,202 | INFO | Successfully loaded faiss with AVX2 support.


/d/hpc/home/ma76193/.local/lib/python3.11/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


2025-12-09 20:44:22,193 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:44:52,194 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:45:22,195 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:45:52,197 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:46:22,198 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:46:52,199 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:47:22,200 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:47:52,201 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:48:22,203 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:48:52,204 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:49:05,901 | INFO | PyTorch version 2.7.0+cu118 available.
2025-12-09 20:49:05,903 | INFO | Duckdb version 1.3.0 available.
2025-12-09 20:49:22,205 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:49:48,305 | INFO | ================================================================================
2025-12-09 20:49:48,306 | INFO | *** BM25 + CROSS-ENCODE

Dec 09, 2025 8:49:49 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


PosixPath('runs/retrieval/notebook.seed1337.rerank3.jsonl')

## 5. LLM Loading: Qwen or Falcon

**To use Falcon-7B-Instruct instead of Qwen:**
```python
USE_FALCON = True  # Change this line
```

**Smart Model Loading:**
- First tries to load from local HF cache
- If corrupted/fails, downloads fresh copy

In [10]:
# ============================================================================
# CELL 9: LLM Configuration with Smart Loading
# ============================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM
import shutil

# ========================================================================
# MODEL SELECTION
# To use Falcon-7B-Instruct: set USE_FALCON = True
# To use Qwen2.5-7B-Instruct (default): set USE_FALCON = False
# ========================================================================
USE_FALCON = False  # <-- CHANGE THIS TO True FOR FALCON

# Similarly for SAFE and MARS:
USE_FALCON_FOR_SAFE = False  # <-- Change to True to use Falcon for SAFE
USE_FALCON_FOR_MARS = False  # <-- Change to True to use Falcon for MARS

QWEN_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
FALCON_MODEL_ID = "tiiuae/falcon-7b-instruct"
MODEL_ID = FALCON_MODEL_ID if USE_FALCON else QWEN_MODEL_ID

def parse_docs_for_prompt(docs):
    texts = []
    for i, d in enumerate(docs, 1):
        try:
            text = json.loads(d["raw"]).get("contents", "")
        except:
            text = d.get("raw", "")
        text = " ".join(text.split())[:1200]
        texts.append(f"[d{i}] {text}")
    return "\n".join(texts)

def build_prompt(query, docs, strict=True):
    policy = "Answer only using the provided documents. If the documents do not contain the answer, say you don't know." if strict else "Prefer the provided documents; if insufficient, you may use general knowledge."
    return f"You are a careful assistant. {policy}\n\nQuestion: {query}\n\nDocuments:\n{parse_docs_for_prompt(docs)}\n\nAnswer:"

def load_llm_smart(model_id, force_download=False):
    """
    Smart model loading:
    1. Try to load from local cache first
    2. If fails, download fresh copy
    
    Args:
        model_id: HuggingFace model ID (e.g., 'tiiuae/falcon-7b-instruct')
        force_download: If True, always download fresh copy
    """
    banner(f"LOADING LLM: {model_id}")
    
    # Determine cache path
    hf_home = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
    model_cache_name = model_id.replace("/", "--")
    model_cache_path = Path(hf_home) / "hub" / f"models--{model_cache_name}"
    
    def try_load(local_files_only=False):
        """Attempt to load model with specified settings."""
        tok = AutoTokenizer.from_pretrained(
            model_id, 
            use_fast=True, 
            trust_remote_code=True,
            local_files_only=local_files_only
        )
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        
        if torch.cuda.is_available():
            mdl = AutoModelForCausalLM.from_pretrained(
                model_id, 
                device_map="auto", 
                torch_dtype=torch.bfloat16, 
                trust_remote_code=True,
                local_files_only=local_files_only
            )
            device = "cuda"
        else:
            mdl = AutoModelForCausalLM.from_pretrained(
                model_id, 
                trust_remote_code=True,
                local_files_only=local_files_only
            )
            device = "cpu"
        return tok, mdl, device
    
    # Step 1: Try loading from local cache (unless force_download)
    if not force_download and model_cache_path.exists():
        log.info(f"Found local cache at {model_cache_path}, attempting to load...")
        try:
            tok, mdl, device = try_load(local_files_only=True)
            log.info(f"✓ Successfully loaded {model_id} from local cache on {device}")
            return tok, mdl, device
        except Exception as e:
            log.warning(f"Local cache load failed: {e}")
            log.info("Cache may be corrupted. Will download fresh copy...")
            # Remove corrupted cache
            try:
                shutil.rmtree(model_cache_path)
                log.info(f"Removed corrupted cache at {model_cache_path}")
            except Exception as rm_err:
                log.warning(f"Could not remove cache: {rm_err}")
    
    # Step 2: Download fresh copy
    log.info(f"Downloading {model_id} from HuggingFace Hub...")
    try:
        tok, mdl, device = try_load(local_files_only=False)
        log.info(f"✓ Successfully downloaded and loaded {model_id} on {device}")
        return tok, mdl, device
    except Exception as e:
        log.error(f"Failed to download model: {e}")
        raise RuntimeError(f"Could not load model {model_id}: {e}")

# Load the selected model
llm_tok, llm_mdl, llm_device = load_llm_smart(MODEL_ID)
print(f"\n[INFO] Using: {MODEL_ID} on {llm_device}")

2025-12-09 20:49:49,896 | INFO | ================================================================================
2025-12-09 20:49:49,897 | INFO | *** LOADING LLM: Qwen/Qwen2.5-7B-Instruct ***
2025-12-09 20:49:49,898 | INFO | ================================================================================
2025-12-09 20:49:49,899 | INFO | Downloading Qwen/Qwen2.5-7B-Instruct from HuggingFace Hub...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2025-12-09 20:49:52,207 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:50:02,651 | INFO | ✓ Successfully downloaded and loaded Qwen/Qwen2.5-7B-Instruct on cuda

[INFO] Using: Qwen/Qwen2.5-7B-Instruct on cuda


## 6. Answer Generation

In [11]:
# ============================================================================
# CELL 10: Generate RAG Answers
# ============================================================================
def generate_answers(retr_path, out_path, tok, mdl, device, model_name, max_new_tokens=256, strict=True):
    banner(f"GENERATE ANSWERS: {model_name}")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    done = set()
    if out_path.exists():
        with open(out_path, "r") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["id"])
                except:
                    pass
        log.info(f"Resume: {len(done)} done")
    new = 0
    with open(retr_path, "r") as fin, open(out_path, "a") as fout:
        for line in fin:
            ex = json.loads(line)
            qid, q, docs = ex["id"], ex["query"], ex["docs"]
            if qid in done:
                continue
            prompt = build_prompt(q, docs, strict=strict)
            enc = tok(prompt, return_tensors="pt", truncation=True, max_length=2048)
            if device == "cuda":
                enc = {k: v.cuda() for k, v in enc.items()}
            with torch.no_grad():
                out = mdl.generate(**enc, do_sample=False, max_new_tokens=max_new_tokens,
                                   eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id or tok.eos_token_id)
            full_text = tok.decode(out[0], skip_special_tokens=True)
            answer = full_text.split("Answer:", 1)[-1].strip() if "Answer:" in full_text else full_text.strip()
            record = {"id": qid, "query": q, "answer": answer, "docs": docs,
                     "meta": {"ts": datetime.now().isoformat(timespec="seconds"), "model": model_name}}
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            new += 1
            log.info(f"[{qid}] len={len(answer)}")
    log.info(f"Done: {new} new")
    return out_path

model_suffix = "falcon7b" if USE_FALCON else "qwen7b"
ANSWERS_FILE = ANS_DIR / f"notebook.seed1337.{model_suffix}.jsonl"
generate_answers(RETR_FILE, ANSWERS_FILE, llm_tok, llm_mdl, llm_device, MODEL_ID)

2025-12-09 20:50:02,666 | INFO | ================================================================================
2025-12-09 20:50:02,667 | INFO | *** GENERATE ANSWERS: Qwen/Qwen2.5-7B-Instruct ***
2025-12-09 20:50:02,667 | INFO | ================================================================================
2025-12-09 20:50:02,671 | INFO | Resume: 50 done
2025-12-09 20:50:02,673 | INFO | Done: 0 new


PosixPath('runs/answers/notebook.seed1337.qwen7b.jsonl')

## 7. SAFE Evaluation

In [ ]:
# ============================================================================
# CELL 11: SAFE Evaluation
# ============================================================================
import importlib.util
import pandas as pd

LOCAL_EVAL_CLAIM = Path("/d/hpc/projects/FRI/ma76193/IR_Project/src/TruthTorchLM/src/TruthTorchLM/long_form_generation/evaluators/eval_claim.py")

def load_safe_evaluator(use_falcon=False):
    banner("INIT SAFE")
    if LOCAL_EVAL_CLAIM.exists():
        spec = importlib.util.spec_from_file_location("safe_eval_claim", LOCAL_EVAL_CLAIM)
        safe_module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(safe_module)
        ClaimEvaluator = safe_module.ClaimEvaluator
    else:
        try:
            from TruthTorchLM.long_form_generation.evaluators.eval_claim import ClaimEvaluator
        except:
            log.error("ClaimEvaluator not found")
            return None
    
    # Use smart loading for SAFE model too
    if use_falcon:
        safe_tok, safe_mdl, _ = load_llm_smart(FALCON_MODEL_ID)
    else:
        safe_tok, safe_mdl = llm_tok, llm_mdl
    
    return ClaimEvaluator(rater_model=safe_mdl, rater_tokenizer=safe_tok, lucene_index_dir=str(FULL_INDEX_DIR), max_steps=3, max_retries=3, bm25_k=3)

def run_safe(answers_file, out_jsonl, out_csv, evaluator):
    banner("RUN SAFE")
    if evaluator is None:
        return
    rows = []
    with open(answers_file, "r") as fin, open(out_jsonl, "w") as fout:
        for line in fin:
            ex = json.loads(line)
            log.info(f"[SAFE] {ex['id']}")
            try:
                res = evaluator(ex["answer"])
                row = {"id": ex["id"], "safe_score": res.get("answer"), "safe_response": res.get("response", ""), "safe_details": res.get("search_details", [])}
            except Exception as e:
                row = {"id": ex["id"], "safe_score": None, "safe_response": str(e), "safe_details": []}
            rows.append(row)
            fout.write(json.dumps(row) + "\n")
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    log.info(f"SAFE done: {out_jsonl}")

safe_eval = load_safe_evaluator(use_falcon=USE_FALCON_FOR_SAFE)
SAFE_JSONL = UE_DIR / "notebook.seed1337.safe.jsonl"
SAFE_CSV = UE_DIR / "notebook.seed1337.safe.csv"
run_safe(ANSWERS_FILE, SAFE_JSONL, SAFE_CSV, safe_eval)

2025-12-09 20:50:03,579 | INFO | ================================================================================
2025-12-09 20:50:03,580 | INFO | *** INIT SAFE ***
2025-12-09 20:50:03,580 | INFO | ================================================================================
2025-12-09 20:50:22,208 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:50:52,209 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:51:22,210 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:51:52,212 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:52:22,213 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:52:52,214 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:53:22,215 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:53:52,217 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:54:22,218 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:54:52,219 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:55:22,220 | INFO | [HEARTBEAT] Notebook alive...
2025-12-09 20:55:52,221 | IN

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[SAFE] Query: "Fernando (footballer, born 1987) Galatasaray SK Turkey join year"
[SAFE] Retrieved 1977 characters of evidence
[SAFE] === Step 2/3 ===
2025-12-09 21:03:22,239 | INFO | [HEARTBEAT] Notebook alive...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[SAFE] Query: "Fernando (footballer, born 1987) Turkish club Galatasaray SK joining year"
[SAFE] Retrieved 1977 characters of evidence
[SAFE] === Step 3/3 ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[SAFE] Query: "Fernando (footballer, born 1987) Turkish club Galatasaray SK joined year"
[SAFE] Retrieved 1977 characters of evidence
[SAFE] Final extracted label = Supported
2025-12-09 21:03:32,147 | INFO | [SAFE] b002

[SAFE] Evaluating claim: Daniel Alexander Cameron (December 10, 1870 – September 4, 1937) was a Canadian  ...
[SAFE] === Step 1/3 ===
2025-12-09 21:03:52,240 | INFO | [HEARTBEAT] Notebook alive...


## 8. MARS (White-box UE)

In [ ]:
# ============================================================================
# CELL 12: MARS Uncertainty Estimation
# ============================================================================
def run_mars(answers_file, out_jsonl, out_csv, use_falcon=False):
    banner("MARS")
    model_id = FALCON_MODEL_ID if use_falcon else QWEN_MODEL_ID
    use_tt = False
    try:
        from TruthTorchLM.uncertainty_estimation import MARS as TT_MARS
        use_tt = True
    except:
        pass
    
    # Use smart loading for MARS model
    if use_falcon:
        tok, mdl, _ = load_llm_smart(model_id)
    else:
        tok, mdl = llm_tok, llm_mdl
    
    rows = []
    with open(answers_file, "r") as f:
        for line in f:
            ex = json.loads(line)
            qid, q, ans, docs = ex["id"], ex["query"], ex["answer"], ex["docs"]
            score, details = None, {}
            if use_tt:
                try:
                    mars_est = TT_MARS(model=mdl, tokenizer=tok)
                    score = float(mars_est.compute(question=q, answer=ans, context="\n".join([get_doc_text(d.get("raw","")) for d in docs])))
                    details = {"method": "truthtorchlm"}
                except:
                    pass
            if score is None:
                prompt = build_prompt(q, docs, strict=True)
                full = prompt + "\n\nAnswer: " + ans
                enc = tok(full, return_tensors="pt", truncation=True, max_length=2048)
                if torch.cuda.is_available():
                    enc = {k: v.cuda() for k, v in enc.items()}
                prompt_ids = tok(prompt + "\n\nAnswer:", return_tensors="pt")["input_ids"][0]
                labels = enc["input_ids"][0].clone()
                labels[:prompt_ids.size(0)] = -100
                with torch.no_grad():
                    out = mdl(**enc, labels=labels.unsqueeze(0))
                    score = -float(out.loss.cpu())
                    details = {"method": "nll_fallback"}
            rows.append({"id": qid, "mars_score": score, "mars_details": details})
            log.info(f"[{qid}] MARS={score:.4f}")
    with open(out_jsonl, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    log.info(f"MARS done: {out_jsonl}")

MARS_JSONL = UE_DIR / "notebook.seed1337.mars.jsonl"
MARS_CSV = UE_DIR / "notebook.seed1337.mars.csv"
run_mars(ANSWERS_FILE, MARS_JSONL, MARS_CSV, use_falcon=USE_FALCON_FOR_MARS)

## 9. Eccentricity (Black-box UE)

In [ ]:
# ============================================================================
# CELL 13: Eccentricity
# ============================================================================
from sentence_transformers import SentenceTransformer
import numpy as np
import csv

def run_eccentricity(answers_file, out_jsonl, out_csv, embed_model="sentence-transformers/all-MiniLM-L6-v2"):
    banner("ECCENTRICITY")
    model = SentenceTransformer(embed_model, device="cuda" if torch.cuda.is_available() else "cpu")
    rows = []
    with open(answers_file, "r") as f:
        for line in f:
            ex = json.loads(line)
            qid, ans, docs = ex["id"], ex["answer"], ex["docs"]
            ctx = [get_doc_text(d.get("raw",""))[:1000] for d in docs if get_doc_text(d.get("raw",""))]
            if not ctx:
                rows.append({"id": qid, "ecc": None, "ecc_z": None})
                continue
            v_ans = model.encode([ans], normalize_embeddings=True)[0]
            v_ctx = model.encode(ctx, normalize_embeddings=True)
            centroid = v_ctx.mean(0)
            centroid = centroid / np.linalg.norm(centroid)
            ecc = 1 - float(np.dot(v_ans, centroid))
            rows.append({"id": qid, "ecc": ecc, "ecc_z": None})
    vals = np.array([r["ecc"] for r in rows if r["ecc"] is not None])
    if len(vals) > 1:
        mu, sd = vals.mean(), max(vals.std(ddof=1), 1e-6)
        for r in rows:
            if r["ecc"] is not None:
                r["ecc_z"] = float((r["ecc"] - mu) / sd)
    with open(out_jsonl, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    with open(out_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["id", "ecc", "ecc_z"])
        w.writeheader()
        w.writerows(rows)
    log.info(f"ECC done: {out_jsonl}")

ECC_JSONL = UE_DIR / "notebook.seed1337.ecc.jsonl"
ECC_CSV = UE_DIR / "notebook.seed1337.ecc.csv"
run_eccentricity(ANSWERS_FILE, ECC_JSONL, ECC_CSV)

## 10. BONUS: Semantic Entropy

A novel UE method that generates multiple answer variants and measures semantic diversity.
Higher entropy = more semantically diverse answers = higher uncertainty.

In [ ]:
# ============================================================================
# CELL 14: BONUS - Semantic Entropy
# ============================================================================
from sklearn.cluster import AgglomerativeClustering
from collections import Counter

def compute_semantic_entropy(query, docs, tok, mdl, embed_model, n_samples=5, temperature=0.7, max_new_tokens=128, sim_thresh=0.85):
    prompt = build_prompt(query, docs, strict=True)
    answers = []
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=1536)
    if torch.cuda.is_available():
        enc = {k: v.cuda() for k, v in enc.items()}
    for _ in range(n_samples):
        with torch.no_grad():
            out = mdl.generate(**enc, do_sample=True, temperature=temperature, top_p=0.9, max_new_tokens=max_new_tokens,
                              eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id or tok.eos_token_id)
        full_text = tok.decode(out[0], skip_special_tokens=True)
        ans = full_text.split("Answer:", 1)[-1].strip() if "Answer:" in full_text else full_text[len(prompt):].strip()
        if ans:
            answers.append(ans)
    if len(answers) < 2:
        return {"semantic_entropy": 0.0, "n_clusters": 1, "n_samples": len(answers)}
    embeddings = embed_model.encode(answers, normalize_embeddings=True)
    clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=1-sim_thresh, metric='cosine', linkage='average')
    labels = clustering.fit_predict(embeddings)
    counts = Counter(labels)
    total = sum(counts.values())
    entropy = -sum((c/total) * np.log2(c/total) for c in counts.values() if c > 0)
    max_ent = np.log2(len(answers)) if len(answers) > 1 else 1.0
    return {"semantic_entropy": float(entropy / max_ent), "raw_entropy": float(entropy), "n_clusters": len(counts), "n_samples": len(answers)}

def run_semantic_entropy(answers_file, out_jsonl, out_csv, tok, mdl, n_samples=5, embed_model_name="sentence-transformers/all-MiniLM-L6-v2"):
    banner("SEMANTIC ENTROPY (BONUS)")
    embed_model = SentenceTransformer(embed_model_name, device="cuda" if torch.cuda.is_available() else "cpu")
    rows = []
    with open(answers_file, "r") as f:
        for line in f:
            ex = json.loads(line)
            log.info(f"[SemEnt] {ex['id']}")
            try:
                res = compute_semantic_entropy(ex["query"], ex["docs"], tok, mdl, embed_model, n_samples)
                rows.append({"id": ex["id"], "semantic_entropy": res["semantic_entropy"], "raw_entropy": res.get("raw_entropy",0), "n_clusters": res["n_clusters"], "n_samples": res["n_samples"]})
            except Exception as e:
                rows.append({"id": ex["id"], "semantic_entropy": None, "raw_entropy": None, "n_clusters": None, "n_samples": None})
    vals = np.array([r["semantic_entropy"] for r in rows if r["semantic_entropy"] is not None])
    if len(vals) > 1:
        mu, sd = vals.mean(), max(vals.std(ddof=1), 1e-6)
        for r in rows:
            if r["semantic_entropy"] is not None:
                r["sem_ent_z"] = float((r["semantic_entropy"] - mu) / sd)
            else:
                r["sem_ent_z"] = None
    with open(out_jsonl, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    log.info(f"SemEnt done: {out_jsonl}")

SEMENT_JSONL = UE_DIR / "notebook.seed1337.semantic_entropy.jsonl"
SEMENT_CSV = UE_DIR / "notebook.seed1337.semantic_entropy.csv"
run_semantic_entropy(ANSWERS_FILE, SEMENT_JSONL, SEMENT_CSV, llm_tok, llm_mdl, n_samples=5)

## 11. Final Report (FIXED)

In [ ]:
# ============================================================================
# CELL 15: Final Report (FIXED - f-string formatting bug resolved)
# ============================================================================

def format_val(val, fmt=".3f"):
    """Helper function to safely format values with NaN handling."""
    if pd.notna(val):
        return f"{val:{fmt}}"
    return "N/A"

def write_final_report(queries_file, answers_file, mars_file, ecc_file, safe_file, sement_file, out_md, out_csv, corr_csv):
    banner("FINAL REPORT")
    
    def load_jsonl(p):
        d = {}
        if p.exists():
            with open(p, "r") as f:
                for line in f:
                    try:
                        obj = json.loads(line)
                        d[obj["id"]] = obj
                    except:
                        pass
        return d
    
    answers = load_jsonl(answers_file)
    mars = load_jsonl(mars_file)
    ecc = load_jsonl(ecc_file)
    safe = load_jsonl(safe_file)
    sement = load_jsonl(sement_file) if sement_file.exists() else {}
    
    rows = []
    for qid, ans_data in answers.items():
        safe_label = safe.get(qid, {}).get("safe_score")
        correctness = 1.0 if safe_label == "Supported" else (0.0 if safe_label == "Not Supported" else np.nan)
        rows.append({
            "id": qid, 
            "query": ans_data.get("query", "")[:80], 
            "safe": safe_label, 
            "correctness": correctness,
            "mars": mars.get(qid, {}).get("mars_score"), 
            "ecc": ecc.get(qid, {}).get("ecc"),
            "ecc_z": ecc.get(qid, {}).get("ecc_z"), 
            "sem_ent": sement.get(qid, {}).get("semantic_entropy")
        })
    
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    
    with open(out_md, "w") as f:
        f.write(f"# RAG+UE Report\n\nGenerated: {datetime.now().isoformat()}\n\n")
        f.write(f"Total: {len(df)}, Supported: {(df['correctness']==1).sum()}, Not Supported: {(df['correctness']==0).sum()}\n\n")
        f.write("| ID | Query | SAFE | MARS | ECC_z | SemEnt |\n|--|--|--|--|--|--|\n")
        for _, r in df.iterrows():
            # FIXED: Use helper function instead of inline conditionals
            mars_str = format_val(r['mars'])
            ecc_str = format_val(r['ecc_z'])
            sem_str = format_val(r.get('sem_ent'))
            f.write(f"| {r['id']} | {str(r['query'])[:40]}... | {r['safe']} | {mars_str} | {ecc_str} | {sem_str} |\n")
    
    corr_cols = ["correctness", "mars", "ecc", "ecc_z"]
    if "sem_ent" in df.columns and df["sem_ent"].notna().any():
        corr_cols.append("sem_ent")
    
    corr_df = df[corr_cols].dropna()
    if len(corr_df) > 2:
        correlations = corr_df.corr(method="pearson")
        correlations.to_csv(corr_csv)
        print("\n" + "="*60 + "\nCORRELATION MATRIX\n" + "="*60)
        print(correlations.to_string())
        print("\n" + "="*60)
        print("KEY INSIGHTS:")
        print("="*60)
        if "mars" in correlations.columns:
            print(f"  MARS vs Correctness: {correlations.loc['correctness', 'mars']:.4f}")
        if "ecc" in correlations.columns:
            print(f"  ECC vs Correctness: {correlations.loc['correctness', 'ecc']:.4f}")
        if "sem_ent" in correlations.columns:
            print(f"  Semantic Entropy vs Correctness: {correlations.loc['correctness', 'sem_ent']:.4f}")
    
    banner("DONE")

FINAL_MD = REPORTS_DIR / "notebook.seed1337.report.md"
FINAL_CSV = REPORTS_DIR / "notebook.seed1337.scores.csv"
CORR_CSV = REPORTS_DIR / "notebook.seed1337.correlations.csv"
write_final_report(SAMPLED_QUERIES, ANSWERS_FILE, MARS_JSONL, ECC_JSONL, SAFE_JSONL, SEMENT_JSONL, FINAL_MD, FINAL_CSV, CORR_CSV)

In [ ]:
# ============================================================================
# CELL 16: Cleanup
# ============================================================================
_stop_hb.set()
print("\n" + "="*80 + "\nPIPELINE COMPLETE\n" + "="*80)
print(f"Model: {MODEL_ID}")
print(f"Answers: {ANSWERS_FILE}")
print(f"SAFE: {SAFE_JSONL}")
print(f"MARS: {MARS_JSONL}")
print(f"ECC: {ECC_JSONL}")
print(f"SemEnt: {SEMENT_JSONL}")
print(f"Report: {FINAL_MD}")

---
## Appendix: Semantic Entropy Explanation

### What is Semantic Entropy?

Semantic Entropy is a novel uncertainty estimation method that measures how **semantically diverse** a model's outputs are when asked the same question multiple times with randomness (temperature sampling).

### The Core Idea

**If a model truly knows the answer**, it will produce semantically similar responses even with randomness.

**If a model is uncertain/guessing**, it will produce semantically different responses each time.

### Algorithm Step-by-Step

1. **Generate N answer samples** (default N=5) using temperature sampling
   - Temperature controls randomness: higher T = more random
   - We use T=0.7 (moderate randomness)

2. **Embed all answers** using a sentence transformer
   - Converts text to 384-dimensional vectors
   - Similar meanings → similar vectors

3. **Cluster by semantic similarity**
   - Uses Agglomerative Clustering with cosine distance
   - Threshold=0.85 means answers >85% similar go in same cluster

4. **Compute entropy from cluster distribution**
   - `H = -Σ(p_i × log₂(p_i))`
   - Where `p_i = (size of cluster i) / (total samples)`

5. **Normalize** by max possible entropy: `H_norm = H / log₂(N)`

### What Does Temperature Mean?

**Temperature** is a parameter that controls the randomness of the model's output:

- **T=0**: Greedy decoding, always picks the most likely token (deterministic)
- **T=0.7**: Moderate randomness, good balance between diversity and coherence
- **T=1.0**: Standard sampling from the probability distribution
- **T>1.0**: More random, less coherent outputs

We use T=0.7 because it's high enough to reveal uncertainty but low enough to produce coherent answers.

### Interpretation

| Semantic Entropy | # Clusters | Interpretation |
|------------------|------------|----------------|
| ~0 | 1 | All answers semantically identical → **Confident** |
| ~0.5 | 2-3 | Some variation → **Moderate uncertainty** |
| ~1.0 | 5 (all different) | Every answer different → **High uncertainty** |

### Why It's Better Than MARS/Eccentricity

| Method | What it Measures | Limitation |
|--------|------------------|------------|
| **MARS** | Token-level probability | A model can be confident about wrong tokens |
| **Eccentricity** | Answer-context distance | Doesn't check if model is consistent |
| **Semantic Entropy** | Answer consistency across samples | Directly measures semantic uncertainty |

### Expected Correlation with Correctness

We expect a **negative correlation**:
- **Low entropy → Correct**: Confident models give consistent, correct answers
- **High entropy → Incorrect**: Uncertain models give inconsistent, often wrong answers

### References

- Kuhn et al. (2023). "Semantic Uncertainty: Linguistic Invariances for Uncertainty Estimation in NLG"
- Lin et al. (2023). "LUQ: Long-text Uncertainty Quantification for LLMs"